# Assignment: Flow Matching for Audio Generation

In this assignment you will implement the **inference and training** code for a pitch-conditioned
flow matching model trained on the [NSynth](https://magenta.tensorflow.org/datasets/nsynth)
dataset. A pretrained model (`pretrained_keyboard.pt`) is provided so you can hear results
immediately and focus on understanding the algorithms.

**Before you begin:** Runtime -> Change runtime type -> **T4 GPU** (required).

---

## What is given to you
| File | Contents |
|---|---|
| `dataset.py` | Spectrogram extraction, normalization, `NSynthSpecDataset` |
| `model.py` | Three model architectures and `build_model_from_config` |
| `pretrained_keyboard.pt` | Pretrained UNet-based flow model (~125k params, keyboard sounds) |

---

## Parts and points
| Part | Task | Points |
|---|---|---|
| 1 | Euler sampling | 2 |
| 2a | Naive velocity scaling (warmup) | 1 |
| 2b | Classifier-Free Guidance (CFG) | 2 |
| 3a | Heun's method | 2 |
| 3b | RK4 | 1 |
| 4a | Timestep Sampling | 0.5 |
| 4b | Flow Loss | 1.5 |
| **Total** | | **10** |
| 4c (bonus) | Fine-tuning on a new instrument | +0.5 |
| 5 (bonus) | Beat the baseline (open-ended) | +0.5 |

---

## Submission

When you are done, **convert this notebook to a Python file** for submission:

```
File -> Download -> Download .py
```

or from the command line:

```bash
jupyter nbconvert --to script assignment.ipynb
```

**Submit the following:**
1. `assignment.py` — your converted notebook
2. (Bonus) `submission_q4.npz` and `model_ft_q4.pt` - Part 4 generated samples and fine-tuned checkpoint
3. (Bonus) `submission_q5.npz` and `model_q5.pt` if attempting Part 5

## Setup

In [1]:
# Install missing deps (torch/torchaudio are pre-installed on Colab)
!pip install -q librosa tqdm

import os, sys, json
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader
import IPython.display as ipd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA RTX A6000


In [2]:
# dataset.py, model.py, and pretrained_keyboard.pt are provided in the assignment data folder.
import os, sys
PROJECT_ROOT = '/mntdatalora/src/Music-Intelligence'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'diffusion_based_music_generation')
sys.path.insert(0, DATA_DIR)

In [3]:
# Load dataset and model utilities
from dataset import NSynthSpecDataset, wav_to_spec, spec_to_audio, FREQ_BINS, TIME_FRAMES, SR
from model  import load_flow_model, save_flow_model, build_model_from_config, FlowModelWrapper, NULL_PITCH

# Load the pretrained model (wrapped for standard diffusion convention: t=1 noise, t=0 data)
CKPT_PATH = os.path.join(DATA_DIR, 'pretrained_keyboard.pt')
model, ckpt = load_flow_model(CKPT_PATH, device=device)

print(f'Model: {ckpt["n_params"]:,} parameters  ({ckpt["n_params"]/1e3:.0f}k)')
print(f'Input shape : (batch, 2, {FREQ_BINS}, {TIME_FRAMES})')
print(f'  2 channels = real + imaginary parts of a 0.5-second complex STFT')
print(f'  freq_bins  = {FREQ_BINS} = n_fft/2 + 1  (n_fft=256)')
print(f'  time_frames= {TIME_FRAMES}')
print(f'NULL_PITCH  : {NULL_PITCH}  (the "no conditioning" token used for CFG)')
print(f'Sample rate : {SR} Hz')
print(f'Time convention: t=1 is noise, t=0 is data')

Model: 125,202 parameters  (125k)
Input shape : (batch, 2, 129, 63)
  2 channels = real + imaginary parts of a 0.5-second complex STFT
  freq_bins  = 129 = n_fft/2 + 1  (n_fft=256)
  time_frames= 63
NULL_PITCH  : 128  (the "no conditioning" token used for CFG)
Sample rate : 16000 Hz
Time convention: t=1 is noise, t=0 is data


---
## Background: Rectified Flow in one page

In class, we studied diffusion models, which have roots in stochastic differential equations (SDEs). In recent years, the popular and highly related *flow-matching* formulation has become increasingly popular, which has its roots in deterministic *Ordinary* differential equations (ODEs). When choosing flow matching with a gaussian distribution, and in our specific choice of optimal transport paths, the math becomes very simple! For those wondering, despite some common confusion on diffusion vs. flow matching, they are fundamentally [the same thing](https://diffusionflow.github.io/).

**Training objective** (Flow Matching with Optimal Transport paths):

Given a data sample $x_0$ (spectrogram), noise $\epsilon \sim \mathcal{N}(0,I)$, and pitch label $p$:
$$x_t = (1-t)\,x_0 + t\,\epsilon  \qquad t \sim U[0,1]$$
At $t=0$ this is pure data; at $t=1$ it is pure noise. The velocity along the straight path is:
$$v^* = \epsilon - x_0 \quad\text{(points from data toward noise)}$$
$$\mathcal{L} = \mathbb{E}_{t,x_0,x_1}\bigl[\|v_\theta(x_t, t, p) - v^*\|^2\bigr]$$

The model $v_\theta(x_t, t, p)$ learns to predict this velocity field. From this field, we can then use any classical ODE solver.

**Key model interface:**
```python
v = model(x, t, pitch)
# x     : (B, 2, FREQ_BINS, TIME_FRAMES)  — current noisy spectrogram
# t     : (B,)  float in [0, 1]           — current time (1=noise, 0=data)
# pitch : (B,)  int  in [0, 127]          — MIDI pitch (or NULL_PITCH=128 for uncond)
# v     : same shape as x                 — predicted velocity (data → noise direction, i.e. ε - x₀)
```

---
## Part 1 — Euler Sampling  `[2 pts]`

Implement the basic Euler ODE solver. Starting from Gaussian noise at $t=1$, take `n_steps` equal
steps of size $\Delta t = 1/n$ toward the data distribution at $t=0$:
$$x_{t-\Delta t} = x_t - v_\theta(x_t,\, t,\, p)\,\Delta t, \qquad t = 1,\; 1{-}\Delta t,\; 1{-}2\Delta t,\; \ldots$$

**Hints:**
- All tensors are already on the right device — don't call `.to()` inside the loop.
- Create the time tensor `t_batch = torch.full((B,), t_val, device=x.device)` before each model call.
- The first step starts at $t = 1$ and the last step starts at $t = 1/n$, arriving at $t = 0$.

In [4]:
def euler_sample(model, x1, pitches, n_steps=50):
    """
    Euler ODE integration from noise (t=1) to data (t=0).

    Parameters
    ----------
    model   : flow model (eval mode, on device)
    x1      : (B, 2, FREQ_BINS, TIME_FRAMES)  initial Gaussian noise
    pitches : (B,) MIDI pitches, dtype=torch.long
    n_steps : number of Euler steps

    Returns
    -------
    x : (B, 2, FREQ_BINS, TIME_FRAMES)  generated spectrograms at t=0
    """
    dt = 1.0 / n_steps
    x = x1.clone()
    B = x.shape[0]
    with torch.no_grad():
        for i in range(n_steps):
            t_val = 1.0 - i * dt
            t_batch = torch.full((B,), t_val, device=x.device)
            v = model(x, t_batch, pitches)
            x = x - v * dt
    return x

In [5]:
# === Sanity Check 1 — do not modify ===
torch.manual_seed(0)
x1_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out1 = euler_sample(model, x1_ag.clone(), p_ag, n_steps=20)

assert out1.shape == (4, 2, FREQ_BINS, TIME_FRAMES), \
    f'Wrong output shape: {out1.shape}'
assert not torch.allclose(out1, x1_ag, atol=1e-3), \
    'Output equals input — did you implement the integration loop?'
assert out1.isfinite().all(), 'Output contains NaN or Inf'
assert out1.std() > 0.05, f'Output std={out1.std():.4f} is suspiciously low'
print(f'\u2713 euler_sample | shape={out1.shape}, mean={out1.mean():.4f}, std={out1.std():.4f}')

✓ euler_sample | shape=torch.Size([4, 2, 129, 63]), mean=-0.0020, std=0.3521


In [6]:
# Listen: C major scale (C4 D4 E4 F4 G4 A4 B4 C5)
NOTE_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

torch.manual_seed(7)
demo_pitches = torch.tensor([60,62,64,65,67,69,71,72], dtype=torch.long, device=device)
demo_noise   = torch.randn(len(demo_pitches), 2, FREQ_BINS, TIME_FRAMES, device=device)

samples_euler = euler_sample(model, demo_noise.clone(), demo_pitches, n_steps=50)

print('Euler samples (no CFG):  notice how pitch identity may be weak at scale=1\n')
for spec, pitch in zip(samples_euler, demo_pitches.tolist()):
    audio = spec_to_audio(spec.cpu())
    audio = audio / (audio.abs().max() + 1e-8)
    name  = NOTE_NAMES[pitch % 12] + str(pitch // 12 - 1)
    print(f'  {name} (MIDI {pitch})')
    display(ipd.Audio(audio.numpy(), rate=SR))

Euler samples (no CFG):  notice how pitch identity may be weak at scale=1

  C4 (MIDI 60)


  D4 (MIDI 62)


  E4 (MIDI 64)


  F4 (MIDI 65)


  G4 (MIDI 67)


  A4 (MIDI 69)


  B4 (MIDI 71)


  C5 (MIDI 72)


---
## Part 2a — Naive Velocity Scaling  `[1 pts]`

Before implementing proper CFG, let's explore a **wrong but instructive** idea:
just multiply the velocity by a scalar `scale` at each step.

$$x_{t-\Delta t} = x_t - \underbrace{v_\theta(x_t, t, p) \cdot \texttt{scale}}_{\text{scaled velocity}} \cdot \Delta t$$

When `scale=1.0` this should be identical to `euler_sample`.  
When `scale>1` the integration "goes faster" — think about what happens when you overshoot.

Try `scale=2.0` and `scale=0.5` and listen to the results. What do you notice?

In [7]:
def naive_scale_sample(model, x1, pitches, n_steps=50, scale=1.0):
    """
    Euler sampling (t=1 -> t=0) with velocity multiplied by `scale`.

    Parameters: same as euler_sample, plus
    scale : float - multiply every velocity prediction by this factor

    Returns: (B, 2, FREQ_BINS, TIME_FRAMES)
    """
    dt = 1.0 / n_steps
    x = x1.clone()
    B = x.shape[0]
    with torch.no_grad():
        for i in range(n_steps):
            t_val = 1.0 - i * dt
            t_batch = torch.full((B,), t_val, device=x.device)
            v = model(x, t_batch, pitches) * scale
            x = x - v * dt
    return x

In [8]:
# === Sanity Check 2a — do not modify ===
torch.manual_seed(0)
x1_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out_s1 = naive_scale_sample(model, x1_ag.clone(), p_ag, n_steps=20, scale=1.0)
out_e  = euler_sample(      model, x1_ag.clone(), p_ag, n_steps=20)
out_s2 = naive_scale_sample(model, x1_ag.clone(), p_ag, n_steps=20, scale=2.0)

assert torch.allclose(out_s1, out_e, atol=1e-5), \
    'naive_scale_sample(scale=1.0) must match euler_sample exactly'
assert not torch.allclose(out_s2, out_e, atol=1e-3), \
    'naive_scale_sample(scale=2.0) should differ from scale=1.0'
print('\u2713 naive_scale_sample | scale=1.0 matches Euler, scale=2.0 differs')

✓ naive_scale_sample | scale=1.0 matches Euler, scale=2.0 differs


In [9]:
# Compare scale=1.0 vs scale=2.0 — what does scaling velocity actually do?
torch.manual_seed(42)
test_pitch  = torch.full((1,), 60, dtype=torch.long, device=device)
test_noise  = torch.randn(1, 2, FREQ_BINS, TIME_FRAMES, device=device)

for scale in [0.5, 1.0, 2.0, 4.0]:
    s = naive_scale_sample(model, test_noise.clone(), test_pitch, n_steps=50, scale=scale)
    audio = spec_to_audio(s[0].cpu())
    audio = audio / (audio.abs().max() + 1e-8)
    print(f'scale={scale}')
    display(ipd.Audio(audio.numpy(), rate=SR))

scale=0.5


scale=1.0


scale=2.0


scale=4.0


---
## Part 2b — Classifier-Free Guidance (CFG)  `[2 pts]`

The correct way to amplify conditioning is to run the model **twice** per step:
once with the actual pitch and once with `NULL_PITCH` (the "no conditioning" token),
then combine:

$$v_{\text{cond}}   = v_\theta(x_t,\, t,\, p)\\ v_{\text{uncond}} = v_\theta(x_t,\, t,\, \texttt{NULL\_PITCH})\\ v = v_{\text{uncond}} + s \cdot (v_{\text{cond}} - v_{\text{uncond}})$$

Notice: at $s=1$, this equals $v_{\text{cond}}$ (standard Euler). At $s=0$ it equals $v_{\text{uncond}}$.
Values $s>1$ extrapolate **beyond** the conditional estimate, sharpening pitch adherence.

The Euler update with CFG is then: $x_{t-\Delta t} = x_t - v \cdot \Delta t$

**Why is this better than naive scaling?**  
Naive scaling changes only the *magnitude* of the velocity (affects "how far" you step), which can lead to severe oversaturation artifacts.  
CFG changes the *direction and magnitude*, as it steers toward regions that are more associated
with the conditioned pitch, without just overshooting.

**Hints:**
- `torch.full_like(pitches, NULL_PITCH)` creates the unconditional pitch batch.
- When `guidance_scale == 1.0`, skip the second model call (just return `v_cond`).

In [10]:
def cfg_sample(model, x1, pitches, n_steps=50, guidance_scale=1.0):
    """
    Euler sampling (t=1 -> t=0) with Classifier-Free Guidance.

    Parameters
    ----------
    model          : flow model
    x1             : (B, 2, FREQ_BINS, TIME_FRAMES)  initial noise
    pitches        : (B,) MIDI pitches, dtype=torch.long
    n_steps        : Euler integration steps
    guidance_scale : float; 1.0 = no guidance, 3-7 = strong

    Returns
    -------
    (B, 2, FREQ_BINS, TIME_FRAMES)
    """
    dt = 1.0 / n_steps
    x = x1.clone()
    B = x.shape[0]
    with torch.no_grad():
        for i in range(n_steps):
            t_val = 1.0 - i * dt
            t_batch = torch.full((B,), t_val, device=x.device)
            v_cond = model(x, t_batch, pitches)
            if guidance_scale == 1.0:
                v = v_cond
            else:
                null_pitches = torch.full_like(pitches, NULL_PITCH)
                v_uncond = model(x, t_batch, null_pitches)
                v = v_uncond + guidance_scale * (v_cond - v_uncond)
            x = x - v * dt
    return x

In [11]:
# === Sanity Check 2b — do not modify ===
torch.manual_seed(0)
x1_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out_cfg1   = cfg_sample(model, x1_ag.clone(), p_ag, n_steps=20, guidance_scale=1.0)
out_euler  = euler_sample(model, x1_ag.clone(), p_ag, n_steps=20)
out_cfg6   = cfg_sample(model, x1_ag.clone(), p_ag, n_steps=20, guidance_scale=6.0)
out_naive2 = naive_scale_sample(model, x1_ag.clone(), p_ag, n_steps=20, scale=2.0)

assert torch.allclose(out_cfg1, out_euler, atol=1e-5), \
    'cfg_sample(guidance_scale=1.0) must equal euler_sample'
assert not torch.allclose(out_cfg6, out_euler, atol=1e-3), \
    'cfg_sample(guidance_scale=6.0) should differ from scale=1.0'
assert not torch.allclose(out_cfg6, out_naive2, atol=1e-3), \
    'CFG (scale=6) must differ from naive scaling (scale=2) — they use different formulas'
print('\u2713 cfg_sample | gs=1.0 matches Euler, gs=6.0 differs, CFG != naive scaling')

✓ cfg_sample | gs=1.0 matches Euler, gs=6.0 differs, CFG != naive scaling


In [12]:
# Listen: same noise, vary guidance scale 1 → 3 → 6 → 10
torch.manual_seed(42)
test_pitch = torch.full((1,), 60, dtype=torch.long, device=device)
test_noise = torch.randn(1, 2, FREQ_BINS, TIME_FRAMES, device=device)

for gs in [1.0, 3.0, 6.0, 10.0]:
    s = cfg_sample(model, test_noise.clone(), test_pitch, n_steps=50, guidance_scale=gs)
    audio = spec_to_audio(s[0].cpu())
    audio = audio / (audio.abs().max() + 1e-8)
    print(f'guidance_scale={gs}  — pitch adherence increases, diversity decreases at high scales')
    display(ipd.Audio(audio.numpy(), rate=SR))

guidance_scale=1.0  — pitch adherence increases, diversity decreases at high scales


guidance_scale=3.0  — pitch adherence increases, diversity decreases at high scales


guidance_scale=6.0  — pitch adherence increases, diversity decreases at high scales


guidance_scale=10.0  — pitch adherence increases, diversity decreases at high scales


---
## Part 3a — Heun's Method  `[2 pts]`

Euler's method has **first-order** local truncation error $O(\Delta t^2)$.  
**Heun's method** (improved Euler / explicit trapezoidal rule) achieves **second-order** accuracy
$O(\Delta t^3)$ by using two velocity evaluations per step:

$$k_1 = v_\theta(x_t,\, t)$$
$$\hat{x} = x_t - k_1 \cdot \Delta t \quad\text{(Euler predictor)}$$
$$k_2 = v_\theta(\hat{x},\, t - \Delta t)$$
$$x_{t-\Delta t} = x_t - \tfrac{1}{2}(k_1 + k_2) \cdot \Delta t \quad\text{(corrected update)}$$

At the **same NFE budget** (number of function evaluations), Heun with $n/2$ steps
uses the same compute as Euler with $n$ steps, but with much lower discretization error.

Your implementation should also work with CFG from the previous part.

In [13]:
def heun_sample(model, x1, pitches, n_steps=50, guidance_scale=1.0):
    """
    Heun's method (2nd-order Runge-Kutta) from t=1 to t=0, with optional CFG.

    Parameters: same as cfg_sample.
    Returns: (B, 2, FREQ_BINS, TIME_FRAMES)
    """
    dt = 1.0 / n_steps
    x = x1.clone()
    B = x.shape[0]
    with torch.no_grad():
        for i in range(n_steps):
            t_val = 1.0 - i * dt
            t_batch = torch.full((B,), t_val, device=x.device)
            k1 = _cfg_velocity(model, x, t_batch, pitches, guidance_scale)
            x_pred = x - k1 * dt
            t_next = torch.full((B,), max(t_val - dt, 0.0), device=x.device)
            k2 = _cfg_velocity(model, x_pred, t_next, pitches, guidance_scale)
            x = x - 0.5 * (k1 + k2) * dt
    return x


def _cfg_velocity(model, x, t_batch, pitches, guidance_scale=1.0):
    v_cond = model(x, t_batch, pitches)
    if guidance_scale == 1.0:
        return v_cond
    null_pitches = torch.full_like(pitches, NULL_PITCH)
    v_uncond = model(x, t_batch, null_pitches)
    return v_uncond + guidance_scale * (v_cond - v_uncond)

In [14]:
# === Sanity Check 3a — do not modify ===
torch.manual_seed(0)
x1_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out_euler = euler_sample(model, x1_ag.clone(), p_ag, n_steps=25)
out_heun  = heun_sample( model, x1_ag.clone(), p_ag, n_steps=25, guidance_scale=1.0)

assert out_heun.shape == (4, 2, FREQ_BINS, TIME_FRAMES), \
    f'Wrong shape: {out_heun.shape}'
assert out_heun.isfinite().all(), 'Heun output contains NaN/Inf'
assert not torch.allclose(out_heun, out_euler, atol=1e-3), \
    'heun_sample must differ from euler_sample'

# CFG version: gs=1 should match no-CFG Heun
out_heun_gs1 = heun_sample(model, x1_ag.clone(), p_ag, n_steps=25, guidance_scale=1.0)
assert torch.allclose(out_heun, out_heun_gs1, atol=1e-5), \
    'heun_sample results should be deterministic given the same inputs'

out_heun_gs6 = heun_sample(model, x1_ag.clone(), p_ag, n_steps=25, guidance_scale=6.0)
assert not torch.allclose(out_heun_gs6, out_heun, atol=1e-3), \
    'heun_sample with guidance_scale=6 should differ from guidance_scale=1'

l2 = (out_heun - out_euler).norm().item()
print(f'\u2713 heun_sample | shape OK, differs from Euler (L2={l2:.3f}), CFG works')

✓ heun_sample | shape OK, differs from Euler (L2=21.447), CFG works


---
## Part 3b — RK4  `[1 pts]`

The **classic 4th-order Runge-Kutta** method uses 4 velocity evaluations per step.
Since we integrate **backward** from $t$ to $t - \Delta t$:

$$k_1 = v_\theta(x_t,\;t)$$
$$k_2 = v_\theta(x_t - k_1\tfrac{\Delta t}{2},\;t - \tfrac{\Delta t}{2})$$
$$k_3 = v_\theta(x_t - k_2\tfrac{\Delta t}{2},\;t - \tfrac{\Delta t}{2})$$
$$k_4 = v_\theta(x_t - k_3\Delta t,\;t - \Delta t)$$
$$x_{t-\Delta t} = x_t - \tfrac{\Delta t}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$


In [15]:
def rk4_sample(model, x1, pitches, n_steps=25, guidance_scale=1.0):
    """
    4th-order Runge-Kutta from t=1 to t=0, with optional CFG.

    Parameters: same as heun_sample.
    Returns: (B, 2, FREQ_BINS, TIME_FRAMES)
    """
    dt = 1.0 / n_steps
    x = x1.clone()
    B = x.shape[0]
    with torch.no_grad():
        for i in range(n_steps):
            t_val = 1.0 - i * dt
            t1 = torch.full((B,), t_val, device=x.device)
            t_mid = torch.full((B,), max(t_val - 0.5 * dt, 0.0), device=x.device)
            t_next = torch.full((B,), max(t_val - dt, 0.0), device=x.device)
            k1 = _cfg_velocity(model, x, t1, pitches, guidance_scale)
            k2 = _cfg_velocity(model, x - k1 * dt * 0.5, t_mid, pitches, guidance_scale)
            k3 = _cfg_velocity(model, x - k2 * dt * 0.5, t_mid, pitches, guidance_scale)
            k4 = _cfg_velocity(model, x - k3 * dt, t_next, pitches, guidance_scale)
            x = x - (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
    return x

In [16]:
# === Sanity Check 3b — do not modify ===
torch.manual_seed(0)
x1_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device)
p_ag  = torch.tensor([60, 62, 64, 67], dtype=torch.long, device=device)

out_rk4 = rk4_sample(model, x1_ag.clone(), p_ag, n_steps=12, guidance_scale=1.0)
assert out_rk4.shape == (4, 2, FREQ_BINS, TIME_FRAMES)
assert out_rk4.isfinite().all()
out_heun = heun_sample(model, x1_ag.clone(), p_ag, n_steps=25, guidance_scale=1.0)
assert not torch.allclose(out_rk4, out_heun, atol=1e-3), 'RK4 should differ from Heun'
print('\u2713 rk4_sample: shape OK, differs from Heun')

✓ rk4_sample: shape OK, differs from Heun


In [17]:
# Compare solvers at equal NFE budget (50 model evaluations each)
# Euler: 50 steps x 1 eval/step = 50 NFE
# Heun:  25 steps x 2 eval/step = 50 NFE
# RK4:   12 steps x 4 eval/step = 48 NFE

torch.manual_seed(7)
pitch1 = torch.full((1,), 60, dtype=torch.long, device=device)
noise1 = torch.randn(1, 2, FREQ_BINS, TIME_FRAMES, device=device)

GS = 6.0  # guidance scale for all

comparisons = [
    ('Euler  (50 steps, gs=6)', lambda: cfg_sample(  model, noise1.clone(), pitch1, n_steps=50,  guidance_scale=GS)),
    ('Heun   (25 steps, gs=6)', lambda: heun_sample( model, noise1.clone(), pitch1, n_steps=25,  guidance_scale=GS)),
    ('RK4    (12 steps, gs=6)', lambda: rk4_sample(  model, noise1.clone(), pitch1, n_steps=12,  guidance_scale=GS)),
]

for label, fn in comparisons:
    s = fn()
    audio = spec_to_audio(s[0].cpu())
    audio = audio / (audio.abs().max() + 1e-8)
    print(label)
    display(ipd.Audio(audio.numpy(), rate=SR))

Euler  (50 steps, gs=6)


Heun   (25 steps, gs=6)


RK4    (12 steps, gs=6)


---
## Part 4a — Timestep Sampling  `[0.5 pts]`

During flow matching training, we sample a random time $t \in [0,1]$ for each example,
controlling *how much* noise is mixed in. Two strategies:

- **Uniform:** $t \sim U[0,1]$ — equal weight across all noise levels
- **Logit-normal:** $t = \sigma(z),\; z \sim \mathcal{N}(0,1)$ — concentrates weight near
  $t = 0.5$ (the hardest denoising regime, used in Stable Diffusion 3 / Flux)

Implement `sample_timesteps` to support both modes.

In [18]:
def sample_timesteps(B: int, device, t_sample: str = 'logit_normal') -> torch.Tensor:
    """
    Sample B timestep values in [0, 1].

    Parameters
    ----------
    B        : batch size
    device   : torch device (e.g. 'cuda' or x.device)
    t_sample : 'uniform' or 'logit_normal'

    Returns
    -------
    t : (B,) tensor of floats in [0, 1]
    """
    if t_sample == 'uniform':
        return torch.rand(B, device=device)
    if t_sample == 'logit_normal':
        return torch.sigmoid(torch.randn(B, device=device))
    raise ValueError("t_sample must be 'uniform' or 'logit_normal'")

In [19]:
# === Sanity Check 4a — do not modify ===
torch.manual_seed(0)
t_unif  = sample_timesteps(1000, device, 'uniform')
t_logit = sample_timesteps(1000, device, 'logit_normal')

assert t_unif.shape  == (1000,), f'Wrong shape: {t_unif.shape}'
assert (t_unif  >= 0).all() and (t_unif  <= 1).all(), 'Uniform t must be in [0, 1]'
assert t_logit.shape == (1000,), f'Wrong shape: {t_logit.shape}'
assert (t_logit >= 0).all() and (t_logit <= 1).all(), 'Logit-normal t must be in [0, 1]'

# logit-normal should be more concentrated near 0.5 than uniform
assert (t_logit - 0.5).abs().mean() < (t_unif - 0.5).abs().mean(), \
    'Logit-normal should concentrate more near t=0.5 than uniform'

print(f'\u2713 sample_timesteps | '
      f'uniform mean|t-0.5|={(t_unif-0.5).abs().mean():.3f}, '
      f'logit-normal mean|t-0.5|={(t_logit-0.5).abs().mean():.3f}')

✓ sample_timesteps | uniform mean|t-0.5|=0.250, logit-normal mean|t-0.5|=0.170


---
## Part 4b — Flow Loss  `[1.5 pts]`

Now implement the core loss function. Given a batch of clean data $x_0$, pitch labels $p$,
and pre-sampled timesteps $t$, your function should:

1. Sample $\epsilon \sim \mathcal{N}(0,I)$ (noise, same shape as $x_0$)
2. Interpolate: $x_t = (1-t)\,x_0 + t\,\epsilon$
3. Target velocity: $v^* = \epsilon - x_0$
4. **CFG dropout:** replace $p$ with `NULL_PITCH` with probability `p_uncond`
5. Predict: $\hat{v} = v_\theta(x_t, t, p)$
6. Return $\mathcal{L} = \text{MSE}(\hat{v},\, v^*)$

**Hints:**
- Broadcast `t` from shape `(B,)` to `(B,2,F,T)` via `t[:,None,None,None]`
- Return `F.mse_loss(v_pred, target)` — **do not call `.item()`** here; the training loop calls `.backward()` on your return value
- CFG dropout: `mask = torch.rand(B, device=x_data.device) < p_uncond`; use `pitch.clone()` to avoid modifying the original

In [20]:
def flow_loss(model, x_data, pitch, t, p_uncond=0.1):
    """
    Compute the flow matching loss for one batch.

    Parameters
    ----------
    model   : flow model (should be in train mode when called)
    x_data  : (B, 2, FREQ_BINS, TIME_FRAMES) - clean data batch (x_0), on device
    pitch   : (B,) - MIDI pitch labels, dtype=torch.long, on device
    t       : (B,) - sampled timestep values in [0, 1], on device
    p_uncond: CFG dropout probability (replace pitch with NULL_PITCH)

    Returns
    -------
    loss : scalar tensor (differentiable - do NOT call .item())
    """
    noise = torch.randn_like(x_data)
    t_view = t[:, None, None, None]
    x_t = (1.0 - t_view) * x_data + t_view * noise
    target = noise - x_data

    pitch_in = pitch.clone()
    if p_uncond > 0:
        mask = torch.rand(pitch_in.shape[0], device=x_data.device) < p_uncond
        pitch_in[mask] = NULL_PITCH

    v_pred = model(x_t, t, pitch_in)
    return F.mse_loss(v_pred, target)

In [21]:
# === Sanity Check 4b — do not modify ===
import copy
torch.manual_seed(42)

x_data_ag = torch.randn(4, 2, FREQ_BINS, TIME_FRAMES, device=device) * 0.5
p_ag      = torch.randint(0, 128, (4,), dtype=torch.long, device=device)
t_ag      = sample_timesteps(4, device, 'logit_normal')

model_ag = copy.deepcopy(model)
opt_ag   = torch.optim.AdamW(model_ag.parameters(), lr=1e-4)

model_ag.train()
loss = flow_loss(model_ag, x_data_ag, p_ag, t_ag, p_uncond=0.0)

assert loss.shape == (), \
    f'flow_loss must return a scalar tensor, got shape {loss.shape}. Do not call .item()!'
assert loss.grad_fn is not None, \
    'flow_loss must return a differentiable tensor for .backward() to work'
assert 0 < loss.item() < 10, f'Loss {loss.item():.4f} outside expected range (0, 10)'

# Verify a full training step works end-to-end
params_before = [p.data.clone() for p in model_ag.parameters()]
opt_ag.zero_grad(set_to_none=True)
loss.backward()
torch.nn.utils.clip_grad_norm_(model_ag.parameters(), 1.0)
opt_ag.step()
model_ag.eval()

changed = any(not torch.allclose(pb, pa)
              for pb, pa in zip(params_before, model_ag.parameters()))
assert changed, 'Model parameters did not change — check the gradient path in flow_loss'

print(f'\u2713 flow_loss | loss={loss.item():.4f}, training step works')

✓ flow_loss | loss=1.0503, training step works


---
## Part 4c — Fine-tuning on a new instrument  `[+0.5 pts bonus]`
### Download fine-tuning data

We use `nsynth-valid` (~1.4 GB), which contains all instrument families.  
Use `instrument_filter` to select your target (default: `'guitar'`).  
Available families: `bass, brass, flute, guitar, keyboard, mallet, organ, reed, string, synth_lead, vocal`

In [22]:
DATA_ROOT = '/content/nsynth'
VALID_DIR = f'{DATA_ROOT}/nsynth-valid/audio'

if not os.path.exists(VALID_DIR):
    print('Downloading nsynth-valid (~1.4 GB)...')
    !mkdir -p {DATA_ROOT}
    !wget -q http://download.magenta.tensorflow.org/datasets/nsynth/nsynth-valid.jsonwav.tar.gz \
           -O /tmp/nsynth-valid.tar.gz
    !tar -xf /tmp/nsynth-valid.tar.gz -C {DATA_ROOT}
    !rm /tmp/nsynth-valid.tar.gz
    print('Done.')
else:
    print('nsynth-valid already present.')

import glob
all_valid = glob.glob(f'{VALID_DIR}/*.wav')
print(f'Total valid files: {len(all_valid):,}')
for family in ['guitar', 'bass', 'flute', 'brass', 'reed', 'keyboard']:
    n = len([f for f in all_valid if os.path.basename(f).startswith(family)])
    print(f'  {family:12s}: {n:4d} files')

nsynth-valid already present.
Total valid files: 2,081
  guitar      : 2081 files
  bass        :    0 files
  flute       :    0 files
  brass       :    0 files
  reed        :    0 files
  keyboard    :    0 files


In [23]:
# -- YOUR CHOICES --------------------------------------------------------------
TARGET_INSTRUMENT = 'guitar'    # Change to any family listed above
FT_MAX_FILES      = 2000        # Number of files to use
FT_EPOCHS         = 300         # Training epochs
FT_LR             = 1e-3        # Learning rate
FT_BATCH_SIZE     = 64
# ----------------------------------------------------------------------------

import copy, time

Q4_EXISTING_CKPT = '/mntdatalora/src/Music-Intelligence/outputs/diffusion_based_music_generation/runs/q4_full_guitar/checkpoints/model_ft.pt'
Q4_EXISTING_HISTORY = '/mntdatalora/src/Music-Intelligence/outputs/diffusion_based_music_generation/runs/q4_full_guitar/history.json'

if os.path.exists(Q4_EXISTING_CKPT):
    model_ft, ckpt_ft = load_flow_model(Q4_EXISTING_CKPT, device=device)
    model_ft.eval()
    if os.path.exists(Q4_EXISTING_HISTORY):
        with open(Q4_EXISTING_HISTORY) as f:
            q4_history = json.load(f)['history']
        print(f'Loaded completed Q4 checkpoint: {Q4_EXISTING_CKPT}')
        print(f'Q4 completed epochs: {len(q4_history)}, final loss={q4_history[-1]["loss"]:.4f}')
    else:
        print(f'Loaded completed Q4 checkpoint: {Q4_EXISTING_CKPT}')
else:
    model_ft, ckpt_ft = load_flow_model(CKPT_PATH, device=device)

    dataset_ft = NSynthSpecDataset(VALID_DIR, instrument_filter=TARGET_INSTRUMENT,
                                   max_files=FT_MAX_FILES)
    loader_ft  = DataLoader(dataset_ft, batch_size=FT_BATCH_SIZE, shuffle=True,
                            drop_last=True, num_workers=0)
    optimizer_ft = torch.optim.AdamW(model_ft.parameters(), lr=FT_LR, weight_decay=1e-4)

    print(f'Fine-tuning on {len(dataset_ft)} {TARGET_INSTRUMENT} files, '
          f'{len(loader_ft)} batches/epoch, {FT_EPOCHS} epochs')

    # -- Training loop - infrastructure is given; your sample_timesteps + flow_loss do the work --
    t0 = time.time()
    for epoch in range(1, FT_EPOCHS + 1):
        model_ft.train()
        epoch_loss = []
        for x_data, pitch in loader_ft:
            x_data = x_data.to(device, non_blocking=True)
            pitch  = pitch.to(device, non_blocking=True)

            t = sample_timesteps(x_data.shape[0], x_data.device)   # your function

            optimizer_ft.zero_grad(set_to_none=True)
            loss = flow_loss(model_ft, x_data, pitch, t, p_uncond=0.1)  # your function
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_ft.parameters(), 1.0)
            optimizer_ft.step()

            epoch_loss.append(loss.item())

        if epoch % max(1, FT_EPOCHS // 10) == 0 or epoch == 1:
            print(f'Epoch {epoch:4d}/{FT_EPOCHS}  loss={np.mean(epoch_loss):.4f}  '
                  f'elapsed={time.time()-t0:.0f}s')

    model_ft.eval()
    print(f'\nFine-tuning done in {(time.time()-t0)/60:.1f} min')

Loaded completed Q4 checkpoint: /mntdatalora/src/Music-Intelligence/outputs/diffusion_based_music_generation/runs/q4_full_guitar/checkpoints/model_ft.pt
Q4 completed epochs: 300, final loss=0.1271


### Generate & submit 100 samples

Choose your best sampler and guidance scale.
**The submission must include the starting noise for each sample** so we can verify reproducibility.

In [24]:
# ── YOUR CHOICES ──────────────────────────────────────────────────────────────
Q4_SAMPLER        = 'heun'   # 'euler' | 'cfg' | 'heun' | 'rk4'
Q4_GUIDANCE_SCALE = 6.0
Q4_N_STEPS        = 50
# ─────────────────────────────────────────────────────────────────────────────

N_SUB = 100
# Spread pitches across 3 octaves (C3–B5)
q4_pitches = torch.tensor(
    [(48 + i % 36) for i in range(N_SUB)], dtype=torch.long, device=device)

torch.manual_seed(0)
q4_noises = torch.randn(N_SUB, 2, FREQ_BINS, TIME_FRAMES, device=device)

model_ft.eval()
q4_samples = []
BATCH = 16

with torch.no_grad():
    for i in range(0, N_SUB, BATCH):
        x0b = q4_noises[i:i+BATCH]
        pb  = q4_pitches[i:i+BATCH]
        if Q4_SAMPLER == 'euler':
            out = euler_sample(model_ft, x0b.clone(), pb, n_steps=Q4_N_STEPS)
        elif Q4_SAMPLER == 'heun':
            out = heun_sample( model_ft, x0b.clone(), pb, n_steps=Q4_N_STEPS,
                               guidance_scale=Q4_GUIDANCE_SCALE)
        elif Q4_SAMPLER == 'rk4':
            out = rk4_sample(  model_ft, x0b.clone(), pb, n_steps=Q4_N_STEPS,
                               guidance_scale=Q4_GUIDANCE_SCALE)
        else:  # cfg
            out = cfg_sample(  model_ft, x0b.clone(), pb, n_steps=Q4_N_STEPS,
                               guidance_scale=Q4_GUIDANCE_SCALE)
        q4_samples.append(out.cpu())

q4_samples = torch.cat(q4_samples)  # (100, 2, F, T)
print(f'Generated {len(q4_samples)} samples')

# Listen to a few
for i in range(3):
    audio = spec_to_audio(q4_samples[i])
    audio = audio / (audio.abs().max() + 1e-8)
    print(f'Sample {i}, pitch={q4_pitches[i].item()}')
    display(ipd.Audio(audio.numpy(), rate=SR))

Generated 100 samples
Sample 0, pitch=48


Sample 1, pitch=49


Sample 2, pitch=50


In [25]:
# Save submission for Q4
os.makedirs('/content', exist_ok=True)

Q4_CKPT_PATH = '/content/model_ft_q4.pt'
save_flow_model(
    model_ft,
    Q4_CKPT_PATH,
    ckpt_ft.get('config', ckpt['config']),
    ckpt_ft.get('n_params', ckpt['n_params']),
    target_instrument=TARGET_INSTRUMENT,
    fine_tune_epochs=FT_EPOCHS,
    fine_tune_max_files=FT_MAX_FILES,
)

np.savez_compressed(
    '/content/submission_q4.npz',
    samples        = q4_samples.numpy().astype(np.float32),
    noises         = q4_noises.cpu().numpy().astype(np.float32),
    pitches        = q4_pitches.cpu().numpy().astype(np.int64),
    guidance_scale = np.array(Q4_GUIDANCE_SCALE, dtype=np.float32),
    n_steps        = np.array(Q4_N_STEPS,        dtype=np.int32),
    sampler        = np.array(Q4_SAMPLER),
)
print('Saved: /content/submission_q4.npz')
print('Saved: /content/model_ft_q4.pt')
print()
print('Submit the following files:')
print('  1. /content/submission_q4.npz')
print('  2. /content/model_ft_q4.pt  (your fine-tuned checkpoint)')
print('  3. This notebook (.ipynb)')

Saved: /content/submission_q4.npz
Saved: /content/model_ft_q4.pt

Submit the following files:
  1. /content/submission_q4.npz
  2. /content/model_ft_q4.pt  (your fine-tuned checkpoint)
  3. This notebook (.ipynb)


---
## Part 5 — Beat the Baseline  `[+0.5 pts bonus]`

**Baseline** (pretrained keyboard model, Heun 25 steps, guidance=6):  
- FD@6 ~ 354  
- Pitch class accuracy ~ 79%

Achieve a **lower FD** or **higher pitch accuracy** using any approach:

| Idea | Notes |
|---|---|
| More fine-tuning epochs | Longer training on same data |
| Different target instrument | Easier task = better FD |
| Better inference | Heun/RK4 vs. Euler; higher guidance scale |
| Mixed training | Fine-tune on multiple families |
| Train from scratch | Use `train_step` with a fresh model |
| Model architecture | Load a larger model config |

Describe your approach in the markdown cell below, then generate and save 100 samples.

### My approach for Part 5

I use the 300-epoch guitar fine-tuned model from Part 4, then improve the inference stage for pitch accuracy. For each requested MIDI pitch, I generate multiple candidates using several solver/guidance settings and multiple noise seeds. Each candidate is scored by harmonic energy around the requested pitch compared with nearby off-target pitches, with penalties for unstable spectra. The final Q5 submission keeps the highest-scoring candidate for each pitch. This directly targets the pitch-accuracy side of the baseline while preserving the required samples/noises/pitches format.

In [26]:
# Part 5 method implementation
# Train a better model (model_q5), then generate q5_samples and q5_noises below.

model_q5 = model_ft
model_q5.eval()

Q5_APPROACH = (
    'Use the 300-epoch Q4 guitar fine-tuned model, then generate multiple '
    'candidates per requested pitch with stronger solver/guidance settings. '
    'Select the candidate with the best harmonic target-pitch score and stable '
    'spectrogram statistics.'
)

Q5_CANDIDATE_SETTINGS = [
    ('heun_50_gs5',  'heun', 50, 5.0),
    ('heun_64_gs6',  'heun', 64, 6.0),
    ('rk4_32_gs6',   'rk4',  32, 6.0),
    ('rk4_50_gs65',  'rk4',  50, 6.5),
]
Q5_NOISE_VARIANTS = 3
Q5_N_FFT = 256


def q5_midi_to_hz(pitch):
    return 440.0 * (2.0 ** ((int(pitch) - 69) / 12.0))


def q5_harmonic_energy(mag_freq, pitch, radius=1):
    bin_hz = SR / Q5_N_FFT
    total = 0.0
    weight_total = 0.0
    harmonic = 1
    while True:
        freq = q5_midi_to_hz(pitch) * harmonic
        if freq >= SR / 2:
            break
        center = int(round(freq / bin_hz))
        if 0 < center < len(mag_freq):
            low = max(0, center - radius)
            high = min(len(mag_freq), center + radius + 1)
            weight = 1.0 / np.sqrt(float(harmonic))
            total += float(mag_freq[low:high].mean()) * weight
            weight_total += weight
        harmonic += 1
    return total / max(weight_total, 1e-8)


def q5_pitch_score(spec, pitch):
    spec = spec.detach().cpu().float()
    mag_freq = torch.sqrt(spec[0].square() + spec[1].square()).mean(dim=-1).numpy()
    target = q5_harmonic_energy(mag_freq, pitch)
    off_target = max(q5_harmonic_energy(mag_freq, pitch + d) for d in [-2, -1, 1, 2])
    target_ratio = target / (float(mag_freq.mean()) + 1e-8)
    margin_ratio = (target - off_target) / (abs(off_target) + 1e-8)
    sample_std = float(spec.std())
    sample_absmax = float(spec.abs().max())
    penalty = max(0.0, 0.12 - sample_std) * 8.0
    penalty += max(0.0, sample_std - 1.55) * 1.5
    penalty += max(0.0, sample_absmax - 18.0) * 0.08
    if not bool(torch.isfinite(spec).all()):
        penalty += 1000.0
    return 2.5 * target_ratio + 1.5 * margin_ratio - penalty


def q5_run_sampler(model, noise, pitches, sampler, n_steps, guidance_scale):
    if sampler == 'heun':
        return heun_sample(model, noise, pitches, n_steps=n_steps, guidance_scale=guidance_scale)
    if sampler == 'rk4':
        return rk4_sample(model, noise, pitches, n_steps=n_steps, guidance_scale=guidance_scale)
    if sampler == 'cfg':
        return cfg_sample(model, noise, pitches, n_steps=n_steps, guidance_scale=guidance_scale)
    return euler_sample(model, noise, pitches, n_steps=n_steps)

print(Q5_APPROACH)

Use the 300-epoch Q4 guitar fine-tuned model, then generate multiple candidates per requested pitch with stronger solver/guidance settings. Select the candidate with the best harmonic target-pitch score and stable spectrogram statistics.


In [27]:
# Generate 100 samples from your best Q5 model
# -- YOUR CHOICES --------------------------------------------------------------
Q5_SAMPLER        = 'pitch_guided_candidate_selection'
Q5_GUIDANCE_SCALE = 6.5
Q5_N_STEPS        = 64
# Use model_q5 (defined in cell above)
# ----------------------------------------------------------------------------

q5_pitches = torch.tensor(
    [(48 + i % 36) for i in range(N_SUB)], dtype=torch.long, device=device)

torch.manual_seed(0)
q5_candidate_pitches = q5_pitches.repeat_interleave(Q5_NOISE_VARIANTS)
q5_candidate_noises = torch.randn(
    len(q5_candidate_pitches), 2, FREQ_BINS, TIME_FRAMES, device=device)

best_samples = [None] * N_SUB
best_noises = [None] * N_SUB
best_scores = [-float('inf')] * N_SUB
q5_selected_settings = [None] * N_SUB
q5_selected_scores = [None] * N_SUB
BATCH = 16

model_q5.eval()
with torch.no_grad():
    for setting_name, sampler_name, n_steps, guidance_scale in Q5_CANDIDATE_SETTINGS:
        generated_parts = []
        for i in range(0, len(q5_candidate_pitches), BATCH):
            x0b = q5_candidate_noises[i:i+BATCH]
            pb = q5_candidate_pitches[i:i+BATCH]
            out = q5_run_sampler(
                model_q5, x0b.clone(), pb,
                sampler=sampler_name, n_steps=n_steps,
                guidance_scale=guidance_scale)
            generated_parts.append(out.cpu())
        generated = torch.cat(generated_parts)

        for candidate_idx in range(len(q5_candidate_pitches)):
            sample_idx = candidate_idx // Q5_NOISE_VARIANTS
            pitch = int(q5_candidate_pitches[candidate_idx].item())
            score = q5_pitch_score(generated[candidate_idx], pitch)
            if score > best_scores[sample_idx]:
                best_scores[sample_idx] = score
                best_samples[sample_idx] = generated[candidate_idx].clone()
                best_noises[sample_idx] = q5_candidate_noises[candidate_idx].detach().cpu().clone()
                q5_selected_settings[sample_idx] = setting_name
                q5_selected_scores[sample_idx] = score

q5_samples = torch.stack(best_samples)  # (100, 2, FREQ_BINS, TIME_FRAMES)
q5_noises = torch.stack(best_noises)    # selected starting noise for every final sample
q5_selected_settings = np.array(q5_selected_settings)
q5_selected_scores = np.array(q5_selected_scores, dtype=np.float32)

print(f'Generated {len(q5_samples)} Q5 samples from {len(q5_candidate_pitches) * len(Q5_CANDIDATE_SETTINGS)} candidates')
print(f'Mean selected pitch score: {q5_selected_scores.mean():.4f}')

Generated 100 Q5 samples from 1200 candidates
Mean selected pitch score: 7.3648


In [28]:
# Save submission for Q5
os.makedirs('/content', exist_ok=True)

Q5_CKPT_PATH = '/content/model_q5.pt'
save_flow_model(
    model_q5,
    Q5_CKPT_PATH,
    ckpt_ft.get('config', ckpt['config']),
    ckpt_ft.get('n_params', ckpt['n_params']),
    q5_approach=Q5_APPROACH,
    q5_candidate_settings=Q5_CANDIDATE_SETTINGS,
    q5_sampler=Q5_SAMPLER,
    q5_n_steps=Q5_N_STEPS,
    q5_guidance_scale=Q5_GUIDANCE_SCALE,
)

np.savez_compressed(
    '/content/submission_q5.npz',
    samples        = q5_samples.numpy().astype(np.float32),
    noises         = q5_noises.cpu().numpy().astype(np.float32),
    pitches        = q5_pitches.cpu().numpy().astype(np.int64),
    guidance_scale = np.array(Q5_GUIDANCE_SCALE, dtype=np.float32),
    n_steps        = np.array(Q5_N_STEPS,        dtype=np.int32),
    sampler        = np.array(Q5_SAMPLER),
    selected_setting = q5_selected_settings,
    selected_score   = q5_selected_scores,
)
print('Saved: /content/submission_q5.npz')
print('Saved: /content/model_q5.pt')
print()
print('Submit:')
print('  1. /content/submission_q5.npz')
print('  2. /content/model_q5.pt')
print('  3. This notebook (.ipynb)')

Saved: /content/submission_q5.npz
Saved: /content/model_q5.pt

Submit:
  1. /content/submission_q5.npz
  2. /content/model_q5.pt
  3. This notebook (.ipynb)


---
## Submission checklist

Before submitting, run **Kernel -> Restart and Run All** to confirm everything executes
from scratch without errors. Then convert your notebook to a `.py` file:

```bash
jupyter nbconvert --to script assignment.ipynb
```

| Item | File | Required for |
|---|---|---|
| Converted notebook | `assignment.py` | All parts |
| Q4 samples + noises (bonus) | `submission_q4.npz` | Part 4 |
| Q4 checkpoint (bonus) | `model_ft_q4.pt` | Part 4 |
| Q5 samples + noises (bonus) | `submission_q5.npz` | Part 5 |
| Q5 checkpoint (bonus) | `model_q5.pt` | Part 5 |